[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/03_modeling_eval/03_modeling_eval_design.ipynb)

# 03 · 建模与评测设计（baseline 阶梯 / 约束驱动选型 / 切片与回归门禁 / A-B 功效 / 离线-在线背离 / 单变量检查）

目标：把「建模与评测该怎么组织」从一段口头原则，变成几个**可以运行、可以断言**的小工具。

本 notebook 你会亲手实现：
1. **baseline 阶梯设计器** —— 把每一级台阶的边际收益算出来，看清楚「投入换来多少价值」
2. **模型选型的约束求解器** —— 先划可行集，再在可行集里排序，而不是直接挑"最强"的
3. **离线评测方案生成器** —— 切片定义 + 基于种子方差的回归门禁，注入一次真实的局部回归并检出它
4. **A/B 样本量与功效计算器** —— 从「基准率+最小可探测效应」反推需要多少流量
5. **离线-在线背离模拟器** —— 用重要性加权把"分布偏移造成的差距"和"其余成因造成的差距"拆开
6. **单变量实验检查器** —— 自动判断一批实验是否违反了「每次只改一个维度」的原则

> 心智模型：**baseline 给对照物，约束给可行集，评测给证据，背离分解给下一轮迭代的方向——这是一个闭环，不是一条直线。**

## 1 · baseline 阶梯设计器

每上一级台阶，多花的成本换来多少价值？把这个「边际收益」算出来，比空口说「肯定有用」更有说服力。

In [ ]:
import numpy as np, math

# (阶梯名, 累计投入天数, 预期指标值) —— 数字是示意性的量级，不是某个真实项目的实测值
LADDER = [
    ('rule_based',           1,  0.55),   # 规则/启发式
    ('simple_model',         3,  0.68),   # 简单模型
    ('pretrained_finetune',  7,  0.80),   # 现成预训练方案微调
    ('custom_arch',         30,  0.84),   # 针对约束定制的方案
]

def marginal_gains(ladder):
    """相邻阶梯之间的 (from, to, 投入增量, 指标增量, 单位投入的收益)，按单位投入收益降序排列。"""
    out = []
    for i in range(1, len(ladder)):
        name0, cost0, val0 = ladder[i - 1]
        name1, cost1, val1 = ladder[i]
        cost_delta = cost1 - cost0
        val_delta = val1 - val0
        gpc = val_delta / cost_delta if cost_delta > 0 else float('inf')
        out.append((name0, name1, cost_delta, val_delta, gpc))
    return sorted(out, key=lambda x: -x[4])

mg = marginal_gains(LADDER)
print(f"{'从':<20} {'到':<22} {'+投入(天)':>10} {'+指标':>8} {'单位投入收益':>12}")
for name0, name1, cd, vd, gpc in mg:
    print(f'{name0:<20} {name1:<22} {cd:>10} {vd:>8.3f} {gpc:>12.4f}')

assert mg[0][:2] == ('rule_based', 'simple_model')          # 单位投入收益最高的永远是最早的台阶
assert mg[-1][:2] == ('pretrained_finetune', 'custom_arch')  # 最后一级台阶的单位投入收益最低
print('\n✅ 边际收益递减清晰可见：从规则到简单模型，每天投入换 0.065 个点；')
print('   从预训练微调到定制方案，每天投入只换 0.0017 个点——这正是"这一步值不值得"的量化依据。')

## 2 · 模型选型的约束求解器

先用硬约束（延迟/显存/数据量）过滤出可行集，再在可行集里按精度排序——而不是先看排行榜再倒推约束。

In [ ]:
CANDIDATES = [
    {'name': 'rule_based',   'latency_ms': 2,  'memory_mb': 50,   'min_samples': 0,      'accuracy': 0.55},
    {'name': 'small_cnn',    'latency_ms': 8,  'memory_mb': 200,  'min_samples': 5000,   'accuracy': 0.72},
    {'name': 'rtmdet_tiny',  'latency_ms': 15, 'memory_mb': 400,  'min_samples': 20000,  'accuracy': 0.80},
    {'name': 'rtdetr_r50',   'latency_ms': 28, 'memory_mb': 1200, 'min_samples': 50000,  'accuracy': 0.84},
    {'name': 'detr_vanilla', 'latency_ms': 45, 'memory_mb': 1600, 'min_samples': 300000, 'accuracy': 0.83},  # 论文精度不低，但延迟和数据量都超预算
]

def filter_feasible(candidates, latency_budget, memory_budget, available_data):
    """硬约束过滤：三条约束任意一条不满足就出局。"""
    return [c for c in candidates
            if c['latency_ms'] <= latency_budget
            and c['memory_mb'] <= memory_budget
            and c['min_samples'] <= available_data]

def rank_candidates(feasible):
    """可行集内部才谈精度排序。"""
    return sorted(feasible, key=lambda c: -c['accuracy'])

feasible = filter_feasible(CANDIDATES, latency_budget=30, memory_budget=1500, available_data=60_000)
names_feasible = [c['name'] for c in feasible]
ranked = rank_candidates(feasible)
names_ranked = [c['name'] for c in ranked]

print('可行集:', names_feasible)
print('按精度排序:', names_ranked)
assert 'detr_vanilla' not in names_feasible, 'detr_vanilla 延迟(45ms>30ms)和数据量(30万>6万)都超预算，必须在过滤这一步就出局'
assert names_ranked[0] == 'rtdetr_r50'
assert names_ranked == ['rtdetr_r50', 'rtmdet_tiny', 'small_cnn', 'rule_based']

print('\n✅ detr_vanilla 精度(0.83)其实很接近榜首，但在可行性过滤这一步就被排除——')
print('   这正是"新颖度陷阱"的反例：先看可行集，"最强"的候选未必在可行集里。')

## 3 · 离线评测方案生成器：切片定义 + 基于种子方差的回归门禁

门禁容差不是拍脑袋定的数字，是「种子方差 × 倍数」算出来的。往一个切片里注入真实的回归，
看回归门禁能不能在其余切片正常波动的情况下，只精准揪出这一个切片。

In [ ]:
def regression_gate(baseline_metrics, new_metrics, k=2.0):
    """baseline_metrics / new_metrics: {切片名: [多个随机种子下的指标值]}（同一模型换种子跑出来的分布）。
    容差 = k 倍的 baseline 种子标准差；new 均值低于 (base 均值 - 容差) 判定为回归。"""
    results = {}
    for name in baseline_metrics:
        base_vals = np.array(baseline_metrics[name])
        new_vals = np.array(new_metrics[name])
        base_mean, new_mean = base_vals.mean(), new_vals.mean()
        sigma = base_vals.std(ddof=1) if len(base_vals) > 1 else 0.0
        tol = k * sigma
        results[name] = {'passed': bool(new_mean >= base_mean - tol),
                          'base_mean': float(base_mean), 'new_mean': float(new_mean), 'tolerance': float(tol)}
    return results

rng = np.random.default_rng(7)
SLICES = ['overall', 'small_object', 'night', 'rare_class']
baseline_metrics = {s: (0.80 + rng.normal(0, 0.005, size=5)).tolist() for s in SLICES}

# 新模型：多数切片小幅提升，但 rare_class 被注入一次明显回归（模拟"改动伤到了某个长尾类别"）
new_metrics = {}
for s in SLICES:
    base_mean_s = np.mean(baseline_metrics[s])
    if s == 'rare_class':
        new_metrics[s] = (base_mean_s - 0.08 + rng.normal(0, 0.005, size=5)).tolist()
    else:
        new_metrics[s] = (base_mean_s + 0.02 + rng.normal(0, 0.005, size=5)).tolist()

gate = regression_gate(baseline_metrics, new_metrics, k=2.0)
for name, r in gate.items():
    flag = '✅ 通过' if r['passed'] else '❌ 回归'
    print(f"{name:<14} base={r['base_mean']:.4f}  new={r['new_mean']:.4f}  容差±{r['tolerance']:.4f}  {flag}")

failed = [s for s, r in gate.items() if not r['passed']]
assert failed == ['rare_class']
assert all(gate[s]['passed'] for s in SLICES if s != 'rare_class')
print('\n✅ 门禁精准揪出被注入回归的 rare_class 切片，其余切片的正常波动没有被误判——')
print('   这就是"容差由种子方差撑起来"的意义：既不会被噪声吓到，也不会把真实回归放过。')

## 4 · A/B 样本量与功效计算器

给定基准率与最小可探测效应，反推需要多少流量——以及「效应减半，样本量要涨到原来几倍」这个免费的判断力。

In [ ]:
Z = {0.10: 1.645, 0.05: 1.960, 0.01: 2.576}    # 双侧 alpha -> z_{alpha/2}
ZP = {0.80: 0.842, 0.90: 1.282, 0.95: 1.645}    # power -> z_beta

def required_sample_size(p, delta, alpha=0.05, power=0.80):
    """两比例 A/B 测试每组所需样本量（正态近似）。p: 基准率；delta: 最小可探测效应。"""
    z_a, z_b = Z[alpha], ZP[power]
    return 2 * (z_a + z_b) ** 2 * p * (1 - p) / (delta ** 2)

n_delta_02 = required_sample_size(p=0.10, delta=0.02)
n_delta_01 = required_sample_size(p=0.10, delta=0.01)
print(f'检测 2 个百分点的效应：每组需要 {n_delta_02:,.0f} 个样本')
print(f'检测 1 个百分点的效应：每组需要 {n_delta_01:,.0f} 个样本（效应减半，样本量约变 {n_delta_01/n_delta_02:.1f} 倍）')
assert abs(n_delta_01 / n_delta_02 - 4.0) < 1e-6   # delta 减半 -> 样本量精确变 4 倍（反比平方关系）

n_power_90 = required_sample_size(p=0.10, delta=0.02, power=0.90)
print(f'把功效从 0.80 提到 0.90：每组需要 {n_power_90:,.0f} 个样本（比 0.80 时多 {n_power_90/n_delta_02 - 1:.0%}）')
assert n_power_90 > n_delta_02

print('\n✅ 记住这个反比平方关系比记住公式本身更有用："要求检测的效应越小，需要的流量指数级增长"。')

## 5 · 离线-在线背离模拟器

离线评测集的场景占比，和真实线上流量的场景占比，几乎从来不一样——先把这个差距量化出来。

In [ ]:
offline_weights = np.array([0.85, 0.10, 0.05])   # 离线评测集：晴天占绝大多数
online_weights  = np.array([0.55, 0.20, 0.25])   # 真实线上流量：夜间/雨天占比高得多
# 模型在离线（全精度 checkpoint）和线上（量化部署后）各场景下的真实表现，量级示意
offline_slice_metric = np.array([0.90, 0.70, 0.55])   # [晴天, 雨天, 夜间]
online_slice_metric  = np.array([0.88, 0.68, 0.50])   # 量化部署后略低于离线（呼应 C60 的训练-部署一致性问题）

offline_metric = float(np.sum(offline_weights * offline_slice_metric))
online_metric  = float(np.sum(online_weights * online_slice_metric))
gap_total = offline_metric - online_metric

print(f'离线整体指标: {offline_metric:.4f}')
print(f'线上整体指标: {online_metric:.4f}')
print(f'背离幅度: {gap_total:.4f}')
assert gap_total > 0.05, '这个场景应该展示出明显的离线-在线背离'
print('\n看起来像"模型上线就变差了"，但这个数字里其实混着两种完全不同的成因——')
print('下面的练习 4 会把它拆开：多少是分布偏移的锅，多少是别的原因（比如量化）的锅。')

## 6 · 单变量实验检查器

每个实验相对基线只应该改一个维度——自动检查一批实验有没有偷偷叠加了多个改动。

In [ ]:
def config_diff(base_cfg, new_cfg):
    """返回所有取值不同的字段: {字段: (base值, new值)}。"""
    return {k: (base_cfg[k], new_cfg[k]) for k in base_cfg if base_cfg.get(k) != new_cfg.get(k)}

def is_single_variable(base_cfg, new_cfg):
    return len(config_diff(base_cfg, new_cfg)) <= 1

BASE_CFG = {'arch': 'rtmdet_tiny', 'lr': 0.01, 'aug': 'default', 'epochs': 100}
EXPERIMENTS = [
    {'arch': 'rtmdet_tiny', 'lr': 0.02,        'aug': 'default', 'epochs': 100},   # 合法：只改 lr
    {'arch': 'rtdetr_r18',  'lr': 0.02,        'aug': 'strong',  'epochs': 100},   # 违规：同时改了三项
    {'arch': 'rtmdet_tiny', 'lr': 0.01,        'aug': 'strong',  'epochs': 100},   # 合法：只改 aug
]

for i, exp in enumerate(EXPERIMENTS):
    diff = config_diff(BASE_CFG, exp)
    ok = is_single_variable(BASE_CFG, exp)
    flag = '✅ 单变量' if ok else f'❌ 违规（同时改了 {len(diff)} 项）'
    print(f'实验 {i}: 改动={list(diff.keys())}  {flag}')

assert is_single_variable(BASE_CFG, EXPERIMENTS[0])
assert not is_single_variable(BASE_CFG, EXPERIMENTS[1])
assert is_single_variable(BASE_CFG, EXPERIMENTS[2])
print('\n✅ 实验 1 一次改了 arch/lr/aug 三项——即便它最后指标最高，也说不清收益归因于哪一项。')

## ✏️ 练习 1：baseline 阶梯的停止点

实现 `stopping_point(ladder, min_gain_per_cost)`：沿着阶梯**按顺序**往上爬
（不是按收益排序，是按阶梯本身的先后顺序），只要相邻两级之间的单位投入收益 `>= min_gain_per_cost` 就继续爬，
第一次低于阈值就停在**前一级**。返回停下来的那一级 `(name, cost, value)`。

In [ ]:
def stopping_point(ladder, min_gain_per_cost):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert stopping_point(LADDER, min_gain_per_cost=0.01) == ('pretrained_finetune', 7, 0.80)
assert stopping_point(LADDER, min_gain_per_cost=0.05) == ('simple_model', 3, 0.68)
assert stopping_point(LADDER, min_gain_per_cost=0.10) == ('rule_based', 1, 0.55)
assert stopping_point(LADDER, min_gain_per_cost=0.0001) == ('custom_arch', 30, 0.84)
for thr in (0.10, 0.05, 0.01, 0.0001):
    print(f'阈值 {thr:<8} -> 停在 {stopping_point(LADDER, thr)}')
print('\n✅ 练习 1 通过：阈值越高（越"抠门"），越早停在便宜的台阶上——这正是预算紧张时该有的行为。')

## ✏️ 练习 2：选型的「够用就行」判断

实现 `cheapest_meeting_floor(feasible, accuracy_floor)`：在可行集里，找出精度 `>= accuracy_floor` 的候选中，
**延迟最低**的那一个（而不是精度最高的那一个）。若没有候选满足精度下限，返回 `None`。

In [ ]:
def cheapest_meeting_floor(feasible, accuracy_floor):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pick = cheapest_meeting_floor(feasible, accuracy_floor=0.75)
assert pick['name'] == 'rtmdet_tiny'          # 精度够(0.80>=0.75)且比 rtdetr_r50 更省延迟
pick_none = cheapest_meeting_floor(feasible, accuracy_floor=0.95)
assert pick_none is None                       # 可行集里没有精度到 0.95 的
pick_low = cheapest_meeting_floor(feasible, accuracy_floor=0.50)
assert pick_low['name'] == 'rule_based'        # 精度门槛很低时，最便宜的规则方案就够用
print('精度下限 0.75 ->', pick['name'], '| 精度下限 0.50 ->', pick_low['name'], '| 精度下限 0.95 ->', pick_none)
print('\n✅ 练习 2 通过：不是"选精度最高的"，是"选够用里最省的"——这是约束驱动选型的另一面。')

## ✏️ 练习 3：A/B 功效反解——最小可探测效应

实现 `min_detectable_effect(n, p, alpha=0.05, power=0.80)`：给定每组样本量 `n` 与基准率 `p`，
反解在给定 `alpha`/`power` 下能探测到的最小效应 `delta`（`required_sample_size` 的反函数）。

In [ ]:
def min_detectable_effect(n, p, alpha=0.05, power=0.80):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
delta_back = min_detectable_effect(n_delta_02, p=0.10)
assert abs(delta_back - 0.02) < 1e-6           # 应该精确还原第 4 节算出的 delta=0.02

mde_5000 = min_detectable_effect(5000, p=0.10)
assert 0.016 < mde_5000 < 0.018
print(f'每组只有 5000 个样本时，最小可探测效应 ≈ {mde_5000:.4f}（约 {mde_5000*100:.1f} 个百分点）')
print(f'反解验证: n={n_delta_02:.0f} 时应能探测 delta=0.02，实际反解得到 {delta_back:.4f}')
print('\n✅ 练习 3 通过：流量不够时，不是"测不出来"，是"测不出那么小的效应"——这句话本身就是加分表达。')

## ✏️ 练习 4：离线-在线背离分解

实现 `gap_decomposition(offline_weights, online_weights, offline_slice_metric, online_slice_metric)`：
返回 `(gap_total, gap_from_shift, gap_residual)`。

- `reweighted_offline_metric`：只把权重换成 `online_weights`，**逐场景表现仍用 `offline_slice_metric`**——
  这一步单纯回答"如果场景占比和线上一样，离线模型该打多少分"。
- `gap_from_shift = offline_metric - reweighted_offline_metric`（分布偏移解释掉的那部分）
- `gap_residual = reweighted_offline_metric - online_metric`（剩下才是模型本身在线上真的变差的部分，比如量化影响）

In [ ]:
def gap_decomposition(offline_weights, online_weights, offline_slice_metric, online_slice_metric):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
gap_total_2, gap_shift, gap_resid = gap_decomposition(
    offline_weights, online_weights, offline_slice_metric, online_slice_metric)

assert abs(gap_total_2 - gap_total) < 1e-9
assert abs(gap_total_2 - (gap_shift + gap_resid)) < 1e-9      # 分解必须精确对得上总差距
assert gap_shift > 0 and gap_resid > 0
shift_share = gap_shift / gap_total_2
print(f'总背离: {gap_total_2:.4f} = 分布偏移贡献 {gap_shift:.4f} + 其余成因贡献 {gap_resid:.4f}')
print(f'分布偏移解释了 {shift_share:.0%} 的背离，剩下 {1-shift_share:.0%} 需要去查量化/硬件等其他原因')
print('\n✅ 练习 4 通过："离线在线不一致"不是一个原因，是好几个原因叠加——分解之后才知道先修哪个。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def stopping_point(ladder, min_gain_per_cost):
    idx = 0
    for i in range(1, len(ladder)):
        cost_delta = ladder[i][1] - ladder[i - 1][1]
        val_delta = ladder[i][2] - ladder[i - 1][2]
        gpc = val_delta / cost_delta if cost_delta > 0 else float('inf')
        if gpc < min_gain_per_cost:
            break
        idx = i
    return ladder[idx]

In [ ]:
# 练习 2 参考答案
def cheapest_meeting_floor(feasible, accuracy_floor):
    ok = [c for c in feasible if c['accuracy'] >= accuracy_floor]
    if not ok:
        return None
    return min(ok, key=lambda c: c['latency_ms'])

In [ ]:
# 练习 3 参考答案
def min_detectable_effect(n, p, alpha=0.05, power=0.80):
    z_a, z_b = Z[alpha], ZP[power]
    return math.sqrt(2 * (z_a + z_b) ** 2 * p * (1 - p) / n)

In [ ]:
# 练习 4 参考答案
def gap_decomposition(offline_weights, online_weights, offline_slice_metric, online_slice_metric):
    offline_metric = float(np.sum(offline_weights * offline_slice_metric))
    online_metric = float(np.sum(online_weights * online_slice_metric))
    reweighted_offline_metric = float(np.sum(online_weights * offline_slice_metric))
    gap_total = offline_metric - online_metric
    gap_from_shift = offline_metric - reweighted_offline_metric
    gap_residual = reweighted_offline_metric - online_metric
    return gap_total, gap_from_shift, gap_residual

---
## 🧪 真实工程胶囊：建模与评测设计文档骨架 + 面试口播要点

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 建模与评测设计文档骨架（面试白板可以照这个顺序展开）
# ══════════════════════════════════════════════════════════════════════
# 1. baseline 阶梯：列出 0-3 级台阶各自的形态、预期投入、预期指标，画出边际收益递减
# 2. 约束清单：延迟/显存/数据量/可维护性/合规，用它们过滤出可行集
# 3. 可行集内排序：按精度或"精度-成本"综合分排，标注被过滤掉的候选及原因
# 4. 特征清单：每个特征标注来源、推理时可得性证明、去掉它掉多少分
# 5. 离线评测方案：切片定义表 + 每个切片的回归门禁容差(k*种子标准差) + 显著性检验方式
# 6. 在线评测计划：影子模式看什么 / 灰度看什么 / A-B 需要多少样本量、多久能跑完
# 7. 离线-在线一致性监控：定期重新校验评测集切片占比是否还匹配线上真实流量
# 8. 迭代节奏：单变量原则 + 实验并发上限 + 与数据闭环产出速度对齐

# ══════════════════════════════════════════════════════════════════════
# B. 面试里最容易被追问的三句话，提前想好怎么答
# ══════════════════════════════════════════════════════════════════════
# Q: "你会直接上最新的架构吗？"
# A: "不会。我会先定一个 baseline 阶梯，看清楚每一级台阶的边际收益，
#     再用延迟/显存/数据量这些硬约束筛出可行集，最后才在可行集里比较精度。"
#
# Q: "怎么保证"不掉点"这件事是真的被检查了？"
# A: "回归门禁的容差不是拍脑袋定的，是种子方差乘以一个倍数；
#     切片一多还要做多重比较校正，否则噪声会制造假阳性回归。"
#
# Q: "离线指标很好，为什么上线效果不一样？"
# A: "背离通常不是一个原因：可能是评测集和线上流量的场景分布不一样(可以重加权分解验证)，
#     可能是量化部署后模型行为变了，也可能是评测集本身被模型过去的决策污染了(反馈环偏差)。
#     先做分解，再决定去修哪一个。"

# ══════════════════════════════════════════════════════════════════════
# C. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 实验设计、种子方差与显著性检验的完整方法          -> C61 模块 01（本课只讲面试组织方式）
# · 数据闭环的回归门禁与边际收益曲线的技术细节        -> C58 模块 05（本课引用其结论）
# · 延迟-精度帕累托前沿的完整选型方法                -> C53 模块 05（本课只讲选型的思考顺序）
# · 离线-在线一致性问题的逐层定位方法（预处理/模型/后处理）-> C60 模块 00（本课只讲评测设计侧的预防）
'''
print(RECIPE)
for token in ['baseline 阶梯', '种子方差', '反馈环偏差', 'C61 模块 01', 'C58 模块 05', 'C60 模块 00']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：设计文档骨架 / 高频追问的标准答法 / 与其他课程的分工边界')

### 小结

- **baseline 和评测互为前提**：没有 baseline，复杂模型的收益是无法验证的断言；没有评测方案，
  baseline 和复杂模型之间无法比较。跳过其中任何一个都是常见的扣分项。
- **baseline 阶梯的价值在于把"这一步投入值不值得"变成一个可以算的数**：单位投入收益会随台阶递减，
  面试里能说出"我预计在哪一级停下来、为什么"，比单纯列出所有台阶更有说服力。
- **模型选型先划可行集，再谈精度排序**：延迟/显存/数据量/可维护性是硬约束，
  精度再高的候选，可行性过滤不过也要出局——这是对抗"新颖度陷阱"的核心方法。
- **离线评测是"切片 + 回归门禁 + 显著性"三件套**：门禁容差由种子方差撑起来，
  切片一多必须做多重比较校正，否则会被自己的评测设计坑。
- **A/B 测试要先算样本量**：效应减半，样本量变 4 倍——这个反比平方关系比记住公式本身更有用。
- **离线-在线背离几乎从不是单一原因**：分布偏移、反馈环偏差、指标代理失配、硬件/精度差异、
  选择性记录经常叠加出现；先分解成分再对症下药，而不是笼统地"重新训练看看"。
- **迭代节奏的真实瓶颈通常不是算力**：是数据闭环的产出速度和评测容量，这一点提前说出来，
  等于提前回答了面试官接下来大概率会问的追问。

模块 03 到此结束——建模与评测设计就位之后，下一站是**模块 04 · 服务、部署与容量估算**，
把这些设计决策落回到真实的延迟预算、显存账本与降级方案上。